# Reproducing Doherty et al. (2025) from dandiset 001444

**Paper.** Doherty DW, Chen L, Smith Y, Wichmann T, Chu H-Y, Lytton WW (2025). Enhanced beta power
emerges from simulated parkinsonian primary motor cortex. *npj Parkinson's Disease* 11, 230.
[doi:10.1038/s41531-025-01070-4](https://doi.org/10.1038/s41531-025-01070-4).

**Claims.** `claims/001444/10.1038-s41531-025-01070-4.yaml`, written from the version of record
(Europe PMC, PMC12325672: main text, legends and Methods; the supplement was not available) and
checked by a second reader. The claims file decides what is tested and how.

**Dandiset.** [DANDI:001444](https://dandiarchive.org/dandiset/001444/0.250805.2002), version
0.250805.2002. The paper's data statement names the dandiset without a version. This is the only
published version, and the draft holds the same 17 assets with the same blobs (data check), so
the pinned version holds what the paper deposited.

**What the dandiset holds.** Four sets of four NetPyNE simulations of a 10,073-cell model of mouse
M1, 4.3 s each: control and parkinsonian, rest and activated, per set. The paper's results come
from set sM1_12-12-2023 (Methods, paragraph 2), files `sub-001_ses-sM1-12-12-2023-01` to `-04`.
Each file holds spike times of the cells that fired, one LFP trace and somatic potentials of 20
(paper's set) or 400 (other sets) PT5B cells. One asset, `sub-001/sub-001.nwb`, is a second copy
of `sub-001_ses-sM1-04-22-2024-01.nwb`.

**Outside inputs.** None. The spectrogram of reading S1 is a copy of NetPyNE 1.0.5's Morlet
method in the helper; it is software the Methods name, not data.

| Claim | Tested |
|---|---|
| fig1-network-size | tested |
| model-pt5b-excitability-decrease | blocked: data_missing (no current-injection simulations) |
| robustness-four-sets | blocked: methods_underspecified (measures reported per set) |
| fig2-control-rest-beta, fig2-control-activated-gamma, fig5-pd-rest-beta, fig5-pd-activated-gamma | tested |
| fig2c-control-pt5b-rate-activated, fig2c-control-it23-rate-activated, fig2c-control-it6-rate-activated | tested |
| control-it5-silenced-activated, control-it4-rate-activated, pt5b-highest-rate | tested |
| control-cv-isi, pd-cv-isi | tested |
| control-subthreshold-oscillations, pd-no-subthreshold-oscillations | blocked: methods_underspecified (no detection rule) |
| control-duty-cycle, pd-duty-cycle | tested |
| fig3a, fig3b, fig6a, fig6b spectrogram claims | tested |
| control-coincident-spikes, pd-coincident-spikes | tested |
| fig4-control-unitary-events, fig7-pd-unitary-events, rest-synchrony-increase | tested |
| fig5c-pd-pt5b-rate-activated, fig5c-pd-it23-rate-activated, pd-it5-silenced-activated, pd-it6-rate-unchanged | tested |
| pd-activated-vm-more-negative | tested |
| rest-pt5b-rate-pd-vs-control, rest-it4-rate-pd-vs-control, rest-it5a-rate-pd-vs-control | tested |
| activated-pt5b-rate-pd-vs-control, activated-it6-rate-pd-vs-control | tested |
| rest-beta-power-order-of-magnitude, activated-20-35hz-power-pd-vs-control | tested |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

import m1sim as m

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

started = time.time()
D = {c: m.load_file(m.PAPER_SET, c) for c in m.CONDITIONS}
OTHER = {(s, c): m.load_file(s, c, vm=False) for s in m.SETS[1:] for c in m.CONDITIONS}
print(f"loaded {len(D) + len(OTHER)} files of dandiset {m.DANDISET_ID} version {m.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")
BLOCKS = m.population_blocks([d["pop"] for d in list(D.values()) + list(OTHER.values())])

## Data check

Each check of the description against the values is stated under its output. The archive's
description of the dandiset says each file holds "a local field potential trace, spike train data
from all 10,073 neurons, and membrane potential traces from 400 of the layer 5B pyramidal tract
(PT5B) neurons", and that simulations last 4.3 s.

In [ ]:
A = m.assets()
listing = pd.DataFrame([{"path": p, "size_MB": round(v[2] / 1e6, 1), "blob": v[1]} for p, v in sorted(A.items())])
print(listing.to_string(index=False))
dup = listing[listing.duplicated("blob", keep=False)]
print("\nAssets sharing one blob (byte-identical files):")
print(dup.to_string(index=False))
draft = requests.get(f"{m.API}/dandisets/{m.DANDISET_ID}/versions/draft/assets/", params={"page_size": 200}).json()["results"]
same = sorted((a["path"], a["blob"]) for a in draft) == sorted((p, v[1]) for p, v in A.items())
print("\nDraft holds the same paths and blobs as the pinned version:", same)
versions = requests.get(f"{m.API}/dandisets/{m.DANDISET_ID}/versions/").json()["results"]
print("Versions on the archive:", [v["version"] for v in versions])
meta = requests.get(f"{m.API}/dandisets/{m.DANDISET_ID}/versions/{m.DANDISET_VERSION}/").json()
print("relatedResource:", meta.get("relatedResource"))
print("assetsSummary measurementTechnique:", [t["name"] for t in meta["assetsSummary"].get("measurementTechnique", [])])

The version holds 17 assets: four sets of four simulations, named by set and number, and
`sub-001/sub-001.nwb`, which is the same blob as `sub-001_ses-sM1-04-22-2024-01.nwb` (a recording
deposited twice; it is counted once and not read again). The archive lists no related resource,
and its summary gives "spike sorting technique" as the measurement technique of a simulation.

In [ ]:
rows = []
for (s, c), d in [((m.PAPER_SET, c), d) for c, d in D.items()] + list(OTHER.items()):
    m1 = [g for g in d["pop"] if g < m.FIRST_INPUT_GID]
    rows.append({"set": s, "file": c, "session_id": d["session_id"], "description": d["description"],
                 "start": d["start"][:19], "units": len(d["pop"]), "M1 units": len(m1),
                 "input units": len(d["pop"]) - len(m1),
                 "spikes": sum(len(t) for t in d["trains"].values()),
                 "first spike": min(t[0] for t in d["trains"].values() if len(t)),
                 "last spike": max(t[-1] for t in d["trains"].values() if len(t)),
                 "silent units": sum(len(t) == 0 for t in d["trains"].values()),
                 "unsorted trains": d["unsorted"], "obs_intervals": d["obs_intervals"].tolist(),
                 "Vm traces": len(d["vm_gids"]), "LFP samples": len(d["lfp"]), "LFP Hz": d["lfp_rate"],
                 "LFP unit": d["lfp_unit"]})
info = pd.DataFrame(rows)
print(info.drop(columns=["obs_intervals"]).to_string(index=False))
print("\nobs_intervals in every file:", {str(x) for x in info.obs_intervals})
print("Expected order of the numbered files (claims file): ", m.CONDITIONS)

The session descriptions confirm the numbering of the claims file: -01 control rest ("quiet
wakefulness"), -02 parkinsonian rest, -03 control activated ("movement"), -04 parkinsonian
activated, in every set. Spike times run from 0 to 4.3 s and are in seconds; no unit has zero
spikes, and every train is sorted. The LFP has 172,000 samples at 40 kHz, 4.3 s. The files of the
paper's set (sM1-12-12-2023) hold 20 PT5B membrane potentials, not 400 as the description says;
the other sets hold 400. Each file holds one LFP trace, "LFP 0", where the model records 11
electrodes (claims file, cfg.recordLFP); no file says at which depth it was taken.

In [ ]:
print("Units in the paper's set, by population (M1 populations, then inputs):")
pops = sorted({p for d in D.values() for p in d["pop"].values()},
              key=lambda p: min(g for d in D.values() for g, q in d["pop"].items() if q == p))
tab = pd.DataFrame({m.CONDITIONS[c]: pd.Series(list(d["pop"].values())).value_counts() for c, d in D.items()}).reindex(pops).fillna(0).astype(int)
print(tab.to_string())
print("\nM1 units per file:", {c: int(tab.loc[m.M1_POPS, m.CONDITIONS[c]].sum()) for c in D})
print("Lowest input gid:", min(g for d in D.values() for g in d["pop"] if g >= m.FIRST_INPUT_GID),
      "; highest M1 gid:", max(g for d in D.values() for g in d["pop"] if g < m.FIRST_INPUT_GID))
print("\nPopulation gid blocks bounded from the gids seen in all 16 files "
      "(start in [low, high], end in [low, high], size min and max):")
bl = pd.DataFrame(BLOCKS, index=["start_low", "start_high", "end_low", "end_high", "n_min", "n_max"]).T
print(bl.to_string())
unseen = sum(BLOCKS[b][1] - BLOCKS[a][2] - 1 for a, b in zip(m.M1_POPS, m.M1_POPS[1:])) + BLOCKS["IT2"][1] + (m.FIRST_INPUT_GID - 1 - BLOCKS["PV6"][2])
print("Observed gid ranges of the populations overlap nowhere:",
      all(BLOCKS[a][2] < BLOCKS[b][1] for a, b in zip(m.M1_POPS, m.M1_POPS[1:])),
      "; gids never seen, between blocks:", unseen, "; sum of smallest sizes + unseen gids:", int(bl.n_min.sum()) + unseen)
same_map = all(D[c]["pop"].get(g, p) == p for c in D for g, p in D["01"]["pop"].items())
print("A gid has the same population in all four files of the set:", same_map)

The units table holds only cells that fired: 2,917 to 4,480 M1 units per file of the paper's set,
not 10,073, and input generators that did not fire are missing too (fewer than 1,000 per input
region). The cell names carry NetPyNE's gids, numbered in one block per population; the inputs
start at gid 10073, so the M1 cells are gids 0 to 10072, which is consistent with 10,073 cells.
The size of each population is bounded from the gids seen in any of the 16 files; for the
populations the claims use the bounds differ by at most 8 cells (IT6; IT2, IT4 and IT5A by 1),
and the rates under both bounds are shown below. PT5B has 1,435 cells and all fired in every file.

In [ ]:
# Clock and labels: upward 0 mV crossings of each membrane potential trace against the spike
# times of the unit with the same gid: the share of unit spikes with a crossing within 0.5 ms, by
# span of time. The paper's set (20 traces per file) and, for comparison, the first 20 traces of
# each file of set sM1-04-22-2024.
rows = []
for c, d in D.items():
    for r in m.vm_match(d["vm"], d["trains"]):
        rows.append({"set": m.PAPER_SET, "file": c, "pop": d["pop"].get(r["gid"]), **r,
                     "Vm min": d["vm"][r["gid"]].min(), "Vm max": d["vm"][r["gid"]].max()})
VM_OTHER = {c: m.load_vm_subset("sM1-04-22-2024", c, 20) for c in m.CONDITIONS}
for c, vmd in VM_OTHER.items():
    d = OTHER[("sM1-04-22-2024", c)]
    for r in m.vm_match(vmd, d["trains"]):
        rows.append({"set": "sM1-04-22-2024", "file": c, "pop": d["pop"].get(r["gid"]), **r,
                     "Vm min": vmd[r["gid"]].min(), "Vm max": vmd[r["gid"]].max()})
vmchk = pd.DataFrame(rows)
print(vmchk.groupby(["set", "file"]).agg(traces=("gid", "size"), all_PT5B=("pop", lambda s: bool((s == "PT5B").all())),
      Vm_spikes=("Vm spikes", "sum"), unit_spikes=("unit spikes", "sum"),
      matched_0_05s=("0-0.5 s", "mean"), matched_05_2s=("0.5-2.0 s", "mean"), matched_2_43s=("2.0-4.3 s", "mean"),
      vm_min=("Vm min", "min"), vm_max=("Vm max", "max")).round(3).to_string())
print("PT5B gids with Vm in the paper's set, same in all four files:", all(D[c]["vm_gids"] == D["01"]["vm_gids"] for c in D),
      D["01"]["vm_gids"])
print("Per-trace share matched in W, paper's set, range:", vmchk[vmchk.set == m.PAPER_SET]["2.0-4.3 s"].min(),
      "to", vmchk[vmchk.set == m.PAPER_SET]["2.0-4.3 s"].max())

Every trace belongs to a PT5B unit of the same gid. In set sM1-04-22-2024 the trace's 0 mV
crossings fall on that unit's spike times throughout the 4.3 s, so there the potentials and the
spikes share one clock and one labelling. In the paper's set they do only at the start: the
crossings match the unit's spikes in the first 0.5 s, less from 0.5 to 2 s, and hardly at all in W
(shares printed above), while the numbers of spikes stay similar. The potentials of the paper's set
therefore come from a run that diverged from the one whose spikes are stored, with the same cells
and the same start; that cannot be established from the files and is recorded as a finding. The
potentials range over the values printed, in mV.

In [ ]:
# LFP against the PT5B spike count histogram: the Results say the LFP and the PT5B SCH oscillate in
# antiphase. Correlation at lags from -50 to 50 ms (2 ms bins) in W; a shared clock puts the
# extreme near lag 0, and the contrast between rest (beta) and activated (gamma) shows in the lag
# of the next opposite extreme.
lags = np.arange(-25, 26)
rows = []
for c, d in D.items():
    _, cnt = m.sch(m.pop_spikes(d, "PT5B"))
    x = d["lfp"][int(m.WINDOW[0] * m.FS):int(m.WINDOW[1] * m.FS)]
    lfp2 = x[: len(cnt) * 80].reshape(len(cnt), 80).mean(axis=1)
    a, b = (cnt - cnt.mean()) / cnt.std(), (lfp2 - lfp2.mean()) / lfp2.std()
    r = np.array([np.mean(a[max(0, -k):len(a) - max(0, k)] * b[max(0, k):len(b) - max(0, -k)]) for k in lags])
    i = np.argmin(r)
    rows.append({"file": m.CONDITIONS[c], "r at lag 0": r[25], "most negative r": r[i], "its lag ms": lags[i] * 2,
                 "most positive r": r.max(), "its lag ms ": lags[np.argmax(r)] * 2})
    # shifted control: LFP shifted by 1 s
    sh = np.roll(b, 500)
    rows[-1]["r at lag 0, LFP shifted 1 s"] = np.mean(a * sh)
print(pd.DataFrame(rows).round(3).to_string(index=False))

In every file the LFP and the PT5B SCH are most strongly anticorrelated at a lag of a few ms, and
the correlation is near zero when the LFP is shifted by 1 s, consistent with the antiphase the
Results describe and with one clock for the LFP and the spikes in W.

In [ ]:
EXP = {}   # expected values of each claim, from the claims file, as (value, tolerance)

def within(x, e):
    return abs(x - e[0]) <= e[1] + 1e-9

def rate_rows(pop, conds):
    """Mean, sd and n of a population's rates in W under R1 (largest and smallest block), R2 and R3."""
    rows = []
    for c in conds:
        r = m.rates(D[c], pop, BLOCKS)
        rmin = m.rates(D[c], pop, BLOCKS, size="min")
        for name, x in [("R1", r["R1"]), ("R1 (smallest block)", rmin["R1"]), ("R2", r["R1"][r["R2_mask"]]),
                        ("R3", r["R1"][r["R3_mask"]])]:
            mu, sd, n = m.summary(x)
            rows.append({"file": c, "condition": m.CONDITIONS[c], "pop": pop, "reading": name,
                         "mean": mu, "sd": sd, "n": n})
    return pd.DataFrame(rows)

def compare(pop, c1, c2):
    """p-values of T1 to T4 for rates of `pop` in files c1 and c2, under R1, R2 and R3. Cells are paired
    by gid (same network in all four files of a set); under R2 and R3 the paired tests use the cells
    kept in both files and the unpaired tests the cells kept in each."""
    a, b = m.rates(D[c1], pop, BLOCKS), m.rates(D[c2], pop, BLOCKS)
    out = []
    for name, ka, kb in [("R1", np.ones(len(a["R1"]), bool), np.ones(len(b["R1"]), bool)),
                         ("R2", a["R2_mask"], b["R2_mask"]), ("R3", a["R3_mask"], b["R3_mask"])]:
        both = ka & kb
        row = {"reading": name, "mean 1": a["R1"][ka].mean() if ka.any() else np.nan,
               "mean 2": b["R1"][kb].mean() if kb.any() else np.nan, "n paired": int(both.sum())}
        if both.sum() > 1:
            p = m.tests(a["R1"][both], b["R1"][both])
            row.update({k: p[k] for k in ["T1 Wilcoxon", "T2 paired t"]})
        if ka.sum() > 1 and kb.sum() > 1:
            p = m.tests(a["R1"][ka], b["R1"][kb], paired_ok=False)
            row.update(p)
        out.append(row)
    return pd.DataFrame(out)

def check_rates(cid, pop, conds, keys, stat=("mean", "sd")):
    t = rate_rows(pop, conds)
    print(t.round(3).to_string(index=False))
    print()
    for reading in ["R1", "R1 (smallest block)", "R2", "R3"]:
        sub = t[t.reading == reading].set_index("file")
        vals, ok = [], []
        for (c, s), e in zip(keys, EXP[cid]):
            v = sub.loc[c, s]
            vals.append(round(float(v), 3))
            ok.append(bool(within(v, e)) if np.isfinite(v) else False)
        print(f"{reading:22s} values {vals}  expected {[e[0] for e in EXP[cid]]}  within tolerance {ok}  all: {all(ok)}")

## Claims

Rates are spikes in W (2.0 to 4.3 s) over 2.3 s. The claims file's rate readings are R1 (all cells
of the population, silent ones at 0), R2 (cells with a spike in W) and R3 (cells with a spike in the
4.3 s simulation, those silent in W at 0). R3 was added to the claims file in answer to a question
(questions/001444.yaml); since the files hold exactly the cells that fired, R3 is the cells the file
holds. The paper does not say which cells enter a mean, so an outcome that holds only under R3 rests
on that reading as an assumption. Test readings T1 to T4 are two-sided; under R2 and R3 the paired
tests use the cells kept in both files.


### fig1-network-size

> The simulated M1 network is composed of 10,073 neurons of seven excitatory pyramidal cell types
> (IT, PT5B, CT) and two interneuron cell types (PV, SOM).

Criterion: count the units of the 15 M1 populations in each file of set sM1_12-12-2023; reproduced
when the count is 10,073 in each.

In [ ]:
counts = {m.CONDITIONS[c]: sum(1 for g, p in d["pop"].items() if p in m.M1_POPS) for c, d in D.items()}
print("M1 units per file:", counts, "; expected 10073 in each:", all(v == 10073 for v in counts.values()))
print("Variant, gid range: lowest input gid", min(g for d in D.values() for g in d["pop"] if g >= m.FIRST_INPUT_GID),
      "so M1 gids are 0 to", m.FIRST_INPUT_GID - 1, "=", m.FIRST_INPUT_GID, "cells; sum of block sizes", int(bl.n_max.sum()))
print("Populations present:", [p for p in m.M1_POPS if p in tab.index], len([p for p in m.M1_POPS if p in tab.index]))

**Result: not reproduced.** The files of set sM1_12-12-2023 hold 4,480, 3,231, 2,919 and 2,917 M1 units, not 10,073, because only cells that fired are stored. The criterion's count fails for that reason; the gids are consistent with the paper, since the input generators start at gid 10073 and the 15 M1 populations are all present.

### model-pt5b-excitability-decrease (blocked)

> In the parkinsonian condition the PT5B neuron model was changed (BK and NaT conductances) so that
> its spike count across current injections fell by 64%, as in 6-OHDA mouse slices.

The dandiset holds only the 16 network simulations (data check); there are no single-cell
current-injection simulations, so the claim is not tested.

In [ ]:
print("Acquisition series in the files: LFP 0 and PT5B membrane potentials only;",
      "files:", len(A), "; any asset outside the four sets:", [p for p in A if "ses-sM1" not in p])

### robustness-four-sets (blocked, measures reported)

> The results hold across four sets of four simulations with different random seeds; set
> sM1_12-12-2023 is a typical example and the other three sets are analysed in Supplementary Table 2.

The claims file gives no measures or values for this claim (Supplementary Table 2 was not
available), so it is blocked. As the criterion allows, the measures of the claims below are
reported for each set, without an outcome.

In [ ]:
rows = []
for s in m.SETS:
    for c in m.CONDITIONS:
        d = D[c] if s == m.PAPER_SET else OTHER[(s, c)]
        pt = [t for g, t in d["trains"].items() if d["pop"][g] == "PT5B"]
        r = np.array([np.count_nonzero((t >= 2.0) & (t < 4.3)) / 2.3 for t in pt])
        cv = m.cv_isi([t[(t >= 2.0) & (t < 4.3)] for t in pt], 1)
        _, cnt = m.sch(m.pop_spikes(d, "PT5B"))
        x = d["lfp"][int(2.0 * m.FS):int(4.3 * m.FS)]
        f1, t1, s1 = m.morlet_spec(x, m.FS)
        rows.append({"set": s, "cond": m.CONDITIONS[c], "PT5B n": len(pt), "PT5B rate": r.mean(), "sd": r.std(ddof=1),
                     "CV mean": cv.mean(), "CV<=0.5 %": 100 * np.mean(cv <= 0.5),
                     "SCH peak Hz": m.welch_peak(cnt, 1 / m.SCH_BIN, 0)[0],
                     "LFP Morlet 14-16 Hz power": m.band_power(f1, s1, (14, 16)).mean()})
rob = pd.DataFrame(rows)
print(rob.round(4).to_string(index=False))
r15 = rob.pivot(index="set", columns="cond", values="LFP Morlet 14-16 Hz power")
print("\nParkinsonian over control 14-16 Hz LFP power at rest, per set:")
print((r15["parkinsonian rest"] / r15["control rest"]).round(1).to_string())

### Dominant frequencies: fig2-control-rest-beta, fig2-control-activated-gamma, fig5-pd-rest-beta, fig5-pd-activated-gamma

> fig2-control-rest-beta: In the control condition at rest, M1 activity is dominated by beta-band
> activity of about 20 Hz, particularly clear in the PT5B spike count histogram.
>
> fig2-control-activated-gamma: In the control condition, the activated state shifts M1 activity to
> gamma-band activity of about 44 Hz.
>
> fig5-pd-rest-beta: In the parkinsonian condition at rest, M1 activity shows focused beta-band
> activity at 15 Hz.
>
> fig5-pd-activated-gamma: In the parkinsonian condition the activated state shifts M1 activity to
> gamma-band activity of about 43 Hz.

Criterion: the frequency of the largest peak of the Welch spectrum (1 s segments) in W of the
PT5B SCH (2 ms bins), searched above 0 Hz and above 5 Hz; the LFP is a variant. The claims file's
electrode readings are E1 (electrode 0) and E2 (mean of 11 electrodes); the files hold one trace,
"LFP 0", taken here as E1, and E2 cannot be computed.

In [ ]:
EXP_PEAK = {"01": (20, 2), "03": (44, 4.4), "02": (15, 1.5), "04": (43, 4.3)}
ID = {"01": "fig2-control-rest-beta", "03": "fig2-control-activated-gamma", "02": "fig5-pd-rest-beta", "04": "fig5-pd-activated-gamma"}
SPEC = {}
rows = []
for c, d in D.items():
    _, cnt = m.sch(m.pop_spikes(d, "PT5B"))
    x = d["lfp"][int(2.0 * m.FS):int(4.3 * m.FS)]
    row = {"claim": ID[c], "condition": m.CONDITIONS[c], "expected": f"{EXP_PEAK[c][0]} +/- {EXP_PEAK[c][1]}"}
    for lo in (0, 5):
        pk, f, p = m.welch_peak(cnt, 1 / m.SCH_BIN, lo)
        row[f"SCH peak >{lo} Hz"] = pk
        row[f"SCH ok >{lo}"] = within(pk, EXP_PEAK[c])
        SPEC[c] = (f, p)
        pk2, f2, p2 = m.welch_peak(x, m.FS, lo)
        row[f"LFP E1 peak >{lo} Hz (variant)"] = pk2
    # the three largest local maxima of the SCH spectrum above 5 Hz, for context
    f, p = SPEC[c]
    sel = (f > 5) & (f <= 100)
    fi, pi = f[sel], p[sel]
    loc = [i for i in range(1, len(pi) - 1) if pi[i] > pi[i - 1] and pi[i] > pi[i + 1]]
    top = sorted(loc, key=lambda i: -pi[i])[:3]
    row["SCH top local maxima Hz (power ratio to largest)"] = [(float(fi[i]), round(float(pi[i] / pi[top[0]]), 2)) for i in top]
    rows.append(row)
peaks = pd.DataFrame(rows)
print(peaks.to_string(index=False))
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for k, c in enumerate(D):
    f, p = SPEC[c]
    ax[k].semilogy(f[(f > 0) & (f <= 100)], p[(f > 0) & (f <= 100)])
    ax[k].set_title(m.CONDITIONS[c]); ax[k].set_xlabel("Hz")
    print(m.CONDITIONS[c], "plotted SCH power range", p[(f > 0) & (f <= 100)].min(), p[(f > 0) & (f <= 100)].max())
plt.tight_layout(); plt.show()

**Result: reproduced.** The largest peak of the control rest PT5B SCH spectrum is at 19 Hz searched above 0 Hz and above 5 Hz, within 18 to 22 Hz. The LFP variant peaks at 1 Hz above 0 Hz and at 19 Hz above 5 Hz.

**Result: reproduced.** The control activated PT5B SCH spectrum peaks at 44 Hz under both search ranges, within 39.6 to 48.4 Hz. The LFP variant peaks at 1 Hz and 13 Hz.

**Result: reproduced.** The parkinsonian rest PT5B SCH spectrum peaks at 15 Hz under both search ranges, within 13.5 to 16.5 Hz; the LFP variant also peaks at 15 Hz.

**Result: not reproduced.** The parkinsonian activated PT5B SCH spectrum peaks at 27 Hz under both search ranges, outside 38.7 to 47.3 Hz; its next local maxima are at 30 and 21 Hz. The LFP variant also peaks at 27 Hz.

### fig2c-control-pt5b-rate-activated

> In the control condition PT5B neurons fire more in the activated state than at rest (11.7 +/- 9.4 against 26.8 +/- 18.2 spikes/s, p < 0.001).

Criterion: files -01 and -03, R1, R2 and R3; reproduced when under one rate reading all four values are within tolerance and the activated rates are higher with p < 0.001 under every test reading.

In [ ]:
EXP["fig2c-control-pt5b-rate-activated"] = [(11.7, 1.17), (9.4, 0.94), (26.8, 2.68), (18.2, 1.82)]
check_rates("fig2c-control-pt5b-rate-activated", "PT5B", ['01', '03'], [('01', 'mean'), ('01', 'sd'), ('03', 'mean'), ('03', 'sd')])
cmpt = compare("PT5B", "01", "03")
print()
print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.001, per reading:", (ps < 0.001).all(axis=1).to_dict())

**Result: reproduced.** Under R1, PT5B rates are 11.69 +/- 9.40 spikes/s at rest and 26.76 +/- 18.17 activated, all four within tolerance, and all four tests give p below 1e-98. R3 is the same, every PT5B cell being in both files; R2 gives 30.60 +/- 16.13 activated, outside tolerance.


### fig2c-control-it23-rate-activated

> In the control condition IT2/3 neurons fire more in the activated state than at rest (1.2 +/- 1.6 against 6.7 +/- 8.9 spikes/s, p < 0.001).

Criterion: files -01 and -03, population IT2, R1, R2 and R3; as above, p < 0.001.

In [ ]:
EXP["fig2c-control-it23-rate-activated"] = [(1.2, 0.12), (1.6, 0.16), (6.7, 0.67), (8.9, 0.89)]
check_rates("fig2c-control-it23-rate-activated", "IT2", ['01', '03'], [('01', 'mean'), ('01', 'sd'), ('03', 'mean'), ('03', 'sd')])
cmpt = compare("IT2", "01", "03")
print()
print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.001, per reading:", (ps < 0.001).all(axis=1).to_dict())

**Result: partially reproduced.** Under R3, IT2 rates are 1.203 +/- 1.649 spikes/s at rest and 6.662 +/- 8.858 activated, all four within tolerance, and the unpaired tests find the increase at p below 0.001 (Mann-Whitney 3.5e-05, Welch 3.6e-06), but the paired tests on the 25 cells kept in both files give 0.016 (Wilcoxon) and 0.011 (paired t), above 0.001. The outcome rests on R3. Under R1 (0.095 +/- 0.565 and 0.262 +/- 2.172) and R2 (1.555 +/- 1.723 and 8.883 +/- 9.220) the values are outside tolerance.


### fig2c-control-it6-rate-activated

> In the control condition IT6 neurons fire less in the activated state than at rest (5.8 +/- 5.8 against 3.5 +/- 5.2 spikes/s, p < 0.001).

Criterion: files -01 and -03, R1, R2 and R3; reproduced when under one reading all four values are within tolerance and the activated rates are lower with p < 0.001 under every test.

In [ ]:
EXP["fig2c-control-it6-rate-activated"] = [(5.8, 0.58), (5.8, 0.58), (3.5, 0.35), (5.2, 0.52)]
check_rates("fig2c-control-it6-rate-activated", "IT6", ['01', '03'], [('01', 'mean'), ('01', 'sd'), ('03', 'mean'), ('03', 'sd')])
cmpt = compare("IT6", "01", "03")
print()
print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.001, per reading:", (ps < 0.001).all(axis=1).to_dict())

**Result: reproduced.** Under R3, IT6 rates are 5.82 +/- 5.78 at rest and 3.47 +/- 5.19 activated, all four within tolerance, and every test finds the decrease at p below 1e-10. The outcome rests on R3: under R1 (4.28 +/- 5.58 and 0.95 +/- 3.13) and R2 (6.27 +/- 5.75 and 4.87 +/- 5.57) some values are outside tolerance, though every test gives p below 0.001 under both.


### control-it5-silenced-activated

> In the control condition IT5A and IT5B neurons fire at about 3 spikes/s at rest (3.3 and 3.1
> spikes/s) and stop firing (0 spikes/s) in the activated state.

Criterion: files -01 and -03, readings R1 and R3; R2 a variant where defined. Tolerance 0.05 spikes/s for
the zero rates.

In [ ]:
def it5(cid, rest, act, exp):
    EXP[cid] = exp
    t = pd.concat([rate_rows("IT5A", [rest, act]), rate_rows("IT5B", [rest, act])])
    print(t.round(3).to_string(index=False))
    for reading in ["R1", "R1 (smallest block)", "R2", "R3"]:
        s = t[t.reading == reading].set_index(["pop", "file"])["mean"]
        vals = [s[("IT5A", rest)], s[("IT5B", rest)], s[("IT5A", act)], s[("IT5B", act)]]
        ok = [bool(within(v, e)) if np.isfinite(v) else False for v, e in zip(vals, exp)]
        print(f"{reading:22s} values {np.round(vals, 3).tolist()} expected {[e[0] for e in exp]} within {ok} all: {all(ok)}")
    for c in (act,):
        for p in ("IT5A", "IT5B"):
            n = sum(1 for g, q in D[c]["pop"].items() if q == p)
            sp = sum(len(t) for g, t in D[c]["trains"].items() if D[c]["pop"][g] == p)
            last = max((t[-1] for g, t in D[c]["trains"].items() if D[c]["pop"][g] == p), default=np.nan)
            print(f"{m.CONDITIONS[c]}: {p} units in the file {n}, spikes in 0-4.3 s {sp}, last spike at {last:.3f} s")
it5("control-it5-silenced-activated", "01", "03", [(3.3, 0.33), (3.1, 0.31), (0, 0.05), (0, 0.05)])

**Result: reproduced.** Under R3 the rest rates are 3.25 (IT5A) and 3.08 spikes/s (IT5B), within tolerance of 3.3 and 3.1, and the activated rates are 0, since the one IT5A and two IT5B units of the activated file fired before 0.05 s and none in W. The outcome rests on R3: under R1 the zeros hold but the rest rates are 1.21 and 1.75, and under R2 3.57 and 3.58 with the activated means undefined.


### control-it4-rate-activated

> In the control condition the IT4 population rate rises from 2.6 spikes/s at rest to 4.7 spikes/s in the activated state.

Criterion: files -01 and -03, R1, R2 and R3; reproduced when under one reading both means are within tolerance. The paper reports no test.

In [ ]:
EXP["control-it4-rate-activated"] = [(2.6, 0.26), (4.7, 0.47)]
check_rates("control-it4-rate-activated", "IT4", ['01', '03'], [('01', 'mean'), ('03', 'mean')])

**Result: reproduced.** Under R3 the IT4 rates are 2.57 (11 cells) at rest and 4.74 spikes/s (95 cells) activated, within tolerance of 2.6 and 4.7. The outcome rests on R3: R1 gives 0.037 and 0.586 and R2 3.14 and 8.18.


### pt5b-highest-rate

> PT5B neurons have the highest mean spike rate of any population in the control rest state and in
> both the parkinsonian rest and activated states.

Criterion: files -01, -02 and -04, mean rates in W of every M1 population, population readings (a)
all 15 and (b) the seven excitatory types; reproduced when PT5B is highest in all three under one
rate reading and both population readings; blocked as methods_underspecified when the population
readings disagree.

In [ ]:
rows = []
for c in ["01", "02", "04"]:
    for pop in m.M1_POPS:
        r = m.rates(D[c], pop, BLOCKS)
        rows.append({"file": c, "pop": pop, "R1": r["R1"].mean(), "R2": r["R1"][r["R2_mask"]].mean() if r["R2_mask"].any() else np.nan,
                     "R3": r["R1"][r["R3_mask"]].mean() if r["R3_mask"].any() else np.nan})
hr = pd.DataFrame(rows)
for reading in ["R1", "R2", "R3"]:
    piv = hr.pivot(index="pop", columns="file", values=reading).reindex(m.M1_POPS)
    print(f"\n{reading} mean rates (spikes/s):")
    print(piv.round(2).T.to_string())
    for label, popset in [("(a) all 15", m.M1_POPS), ("(b) excitatory", m.EXCITATORY)]:
        top = {m.CONDITIONS[c]: piv.loc[popset, c].idxmax() for c in piv.columns}
        print(f"  {label}: highest {top}; PT5B highest in all three: {all(v == 'PT5B' for v in top.values())}")

**Result: blocked (methods_underspecified), as the criterion directs when the population readings disagree.** Among the seven excitatory types PT5B has the highest mean rate in all three files under R1, R2 and R3. Among all 15 populations PV5A is higher in all three under every reading (R1 22.51, 22.79 and 42.10 against PT5B 11.69, 14.24 and 29.68 spikes/s; R3 28.83, 32.08 and 58.53, PT5B the same as under R1), and PV5B too under R1 and R3.


### control-cv-isi and pd-cv-isi

> control-cv-isi: In the control condition PT5B interspike-interval CV is highly variable at rest
> (0.59 +/- 0.37; 47.8% regular with CV <= 0.5, 52.2% irregular) and mainly irregular in the
> activated state (0.61 +/- 0.30; 25.7% regular, 74.3% irregular).
>
> pd-cv-isi: In the parkinsonian condition PT5B spiking is mainly regular at rest (CV 0.34 +/- 0.27;
> 80.6% with CV <= 0.5) and more regular in the activated state (CV 0.18 +/- 0.19; 93.8% regular).

Criterion: CV of each PT5B cell's ISIs; mean and standard deviation over cells, share with CV <=
0.5; readings window (a) W or (b) 0 to 4.3 s, cells (i) with at least two ISIs or (ii) at least one.
The ISI-CV statistic of NetPyNE 1.0.5 (plotSpikeStats, cells with more than 10 spikes in the time
range) is shown as a variant (iii).

In [ ]:
EXP["control-cv-isi"] = [(0.59, 0.059), (0.37, 0.037), (47.8, 4.78), (0.61, 0.061), (0.30, 0.030), (25.7, 2.57)]
EXP["pd-cv-isi"] = [(0.34, 0.034), (0.27, 0.027), (80.6, 8.06), (0.18, 0.018), (0.19, 0.019), (93.8, 9.38)]
CV = {}
for cid, (rest, act) in {"control-cv-isi": ("01", "03"), "pd-cv-isi": ("02", "04")}.items():
    print(cid)
    for wname, w in [("(a) W", m.WINDOW), ("(b) 0-4.3 s", (0.0, 4.3))]:
        for cname, k in [("(i) >=2 ISIs", 2), ("(ii) >=1 ISI", 1), ("(iii) >10 spikes, variant", 10)]:
            vals = []
            for c in (rest, act):
                pt = [t[(t >= w[0]) & (t < w[1])] for g, t in D[c]["trains"].items() if D[c]["pop"][g] == "PT5B"]
                cv = m.cv_isi(pt, k)
                vals += [cv.mean(), cv.std(ddof=1), 100 * np.mean(cv <= 0.5)]
                n = len(cv)
            ok = [bool(within(v, e)) for v, e in zip(vals, EXP[cid])]
            CV[(cid, wname, cname)] = vals
            print(f"  {wname:12s} {cname:26s} {np.round(vals, 3).tolist()}  within {ok} all: {all(ok)}  (n cells activated {n})")
    print("  expected", [e[0] for e in EXP[cid]])

**Result: reproduced.** In W with cells of at least one ISI, the PT5B CV is 0.590 +/- 0.365 with 47.8% at or below 0.5 at rest and 0.610 +/- 0.301 with 25.7% activated; with at least two ISIs, 0.611 +/- 0.354 (45.9%) and 0.618 +/- 0.295 (24.7%). All six values are within tolerance under both. Over the whole 4.3 s no value is within tolerance.

**Result: reproduced.** In W with cells of at least one ISI, the parkinsonian PT5B CV is 0.339 +/- 0.267 with 80.6% at or below 0.5 at rest and 0.184 +/- 0.191 with 93.8% activated; with at least two ISIs, 0.343 +/- 0.267 (80.4%) and 0.185 +/- 0.191 (93.8%). All six are within tolerance under both readings in W.

### control-subthreshold-oscillations and pd-no-subthreshold-oscillations (blocked)

> control-subthreshold-oscillations: In the control condition subthreshold membrane-potential
> oscillations appear in 57.4% of PT5B neurons in the activated state and not at rest.
>
> pd-no-subthreshold-oscillations: No subthreshold membrane-potential oscillations are seen in PT5B
> neurons in the parkinsonian rest or activated states.

The claims file finds no detection rule in the paper or its code and blocks both claims as
methods_underspecified; it asks for the number of PT5B cells with a recorded potential.

In [ ]:
for c, d in D.items():
    print(m.CONDITIONS[c], ": PT5B cells with a membrane potential", len(d["vm_gids"]), "of", BLOCKS["PT5B"][5])
print("57.4% of 20 cells would be", 0.574 * 20, "cells; of 1435,", round(0.574 * 1435, 1))

### pd-activated-vm-more-negative

> In the activated state, PT5B membrane potential is on average more negative in the parkinsonian
> than in the control condition, which the paper gives as the reason the EPSPs look larger.

Criterion: files -03 and -04, the PT5B cells recorded in both, in W; readings (a) the mean of each
trace and (b) its median; reproduced when the mean over cells is lower in the parkinsonian file
under both.

In [ ]:
rows = []
a, b = int(2.0 * m.FS), int(4.3 * m.FS)
for g in D["03"]["vm_gids"]:
    if g in D["04"]["vm"]:
        v3, v4 = D["03"]["vm"][g][a:b], D["04"]["vm"][g][a:b]
        rows.append({"gid": g, "ctrl mean": v3.mean(), "pd mean": v4.mean(), "ctrl median": np.median(v3), "pd median": np.median(v4),
                     "ctrl spikes": len(m.vm_spikes(v3)), "pd spikes": len(m.vm_spikes(v4))})
vm = pd.DataFrame(rows)
vm["diff mean"] = vm["pd mean"] - vm["ctrl mean"]
vm["diff median"] = vm["pd median"] - vm["ctrl median"]
print(vm.round(2).to_string(index=False))
from scipy import stats
# Variant: the first 40 PT5B traces of files -03 and -04 of the other three sets, whose potentials
# match their spikes (data check); not the paper's set.
var = []
for s_ in m.SETS[1:]:
    v3 = m.load_vm_subset(s_, "03", 40, m.WINDOW)
    v4 = m.load_vm_subset(s_, "04", 40, m.WINDOW)
    g = [k for k in v3 if k in v4]
    dmean = np.array([v4[k].mean() - v3[k].mean() for k in g])
    dmed = np.array([np.median(v4[k]) - np.median(v3[k]) for k in g])
    var.append({"set": s_, "cells": len(g), "mean diff mV": dmean.mean(), "lower (mean)": int((dmean < 0).sum()),
                "median diff mV": dmed.mean(), "lower (median)": int((dmed < 0).sum())})
print("\nVariant, other sets:")
print(pd.DataFrame(var).round(2).to_string(index=False))
for k in ["mean", "median"]:
    print(f"({'a' if k == 'mean' else 'b'}) {k}: control {vm['ctrl ' + k].mean():.2f} mV, parkinsonian {vm['pd ' + k].mean():.2f} mV, "
          f"difference {vm['diff ' + k].mean():.2f} mV; cells lower in parkinsonian {(vm['diff ' + k] < 0).sum()} of {len(vm)}; "
          f"Wilcoxon p = {stats.wilcoxon(vm['diff ' + k]).pvalue:.3g}")

**Result: reproduced.** In W the mean PT5B potential over the 20 recorded cells is -60.84 mV in the control and -66.46 mV in the parkinsonian activated file (difference -5.62 mV), and the median -63.98 against -71.11 mV; every one of the 20 cells is lower in the parkinsonian file under both readings. The potentials of this set do not match its stored spikes in W (data check); in the other three sets, whose potentials do, the parkinsonian potential is lower in 39, 39 and 40 of 40 cells by the mean (by 4.98 to 5.67 mV) and in 40 of 40 by the median.

### control-duty-cycle and pd-duty-cycle

> control-duty-cycle: In the control condition the PT5B duty cycle, the share of each oscillation
> period with activity above half-height, is 31% at rest and 48% in the activated state.
>
> pd-duty-cycle: In the parkinsonian condition the PT5B duty cycle is 29% at rest and 25% in the
> activated state.

Criterion: PT5B SCH, 2 ms bins, unsmoothed; periods from one rising crossing of half-height to the
next; readings half-height (a) halfway between the minimum and maximum of the window or (b) of
each period; window (i) 2.0 to 2.25 s or (ii) W; summary (x) mean of per-period duty cycles or (y)
total time above over total time of complete periods. Reproduced when under one combination both
values of a claim are within tolerance.

In [ ]:
EXP["control-duty-cycle"] = [(31, 3.1), (48, 4.8)]
EXP["pd-duty-cycle"] = [(29, 2.9), (25, 2.5)]
DUTY = {}
for cid, (rest, act) in {"control-duty-cycle": ("01", "03"), "pd-duty-cycle": ("02", "04")}.items():
    print(cid, "expected", [e[0] for e in EXP[cid]])
    for hname, local in [("(a) window", False), ("(b) period", True)]:
        for wname, w in [("(i) 2.0-2.25 s", (2.0, 2.25)), ("(ii) W", m.WINDOW)]:
            res = [m.duty_cycles(m.sch(m.pop_spikes(D[c], "PT5B"), w)[1], local) for c in (rest, act)]
            for sname, key in [("(x) mean", "mean"), ("(y) total", "total")]:
                vals = [r[key] for r in res]
                ok = [bool(within(v, e)) for v, e in zip(vals, EXP[cid])]
                DUTY[(cid, hname, wname, sname)] = vals
                print(f"  {hname:11s} {wname:15s} {sname:10s} {np.round(vals, 2).tolist()} within {ok} both: {all(ok)}"
                      f"   periods {[r['n_periods'] for r in res]}, median period ms {[r['median_period_ms'] for r in res]}")

**Result: not reproduced.** No combination of readings puts both control values within tolerance. The closest, global half-height in 2.0 to 2.25 s with the total summary, gives 27.88% at rest (tolerance 27.9 to 34.1) and 46.49% activated (within). Over W the unsmoothed 2 ms SCH crosses half-height every 10 ms (median period), far shorter than the beta and gamma periods.

**Result: not reproduced.** No combination puts both parkinsonian values within tolerance; global half-height in 2.0 to 2.25 s with the total summary gives 26.42% at rest (within 26.1 to 31.9) but 36.45% activated (tolerance 22.5 to 27.5), and every other combination gives 34% or more for the activated state.

### LFP spectrograms: fig3a, fig3b, fig6a, fig6b

> fig3a-control-rest-spectrogram: In the control rest state the LFP spectrogram shows virtually
> continuous power around 2 Hz and 15 to 25 Hz beta bursts recurring every 600 ms (1.7 Hz).
>
> fig3b-control-activated-spectrogram: In the control activated state the 2 Hz activity disappears,
> 15 to 20 Hz power is no longer visible except 15 Hz bursts of about 100 to 500 ms at about 1.5 Hz,
> and brief (about 50 ms) 25 to 35 Hz bursts occur at about 1 Hz.
>
> fig6a-pd-rest-spectrogram: In the parkinsonian rest state the LFP spectrogram shows continuous
> high-power 15 Hz beta with 25 to 35 Hz bursts of about 50 to 250 ms recurring about every 200 ms
> (about 5 Hz).
>
> fig6b-pd-activated-spectrogram: In the parkinsonian activated state the LFP spectrogram shows 20
> to 35 Hz bursts of about 25 to 100 ms recurring every 50 to 100 ms (about 10 to 20 Hz).

Readings S1 (NetPyNE 1.0.5 Morlet spectrogram, its defaults) and S2 (short-time Fourier transform),
on "LFP 0" (E1; E2 cannot be computed). NetPyNE's own fft option with its defaults (256 samples at
40 kHz) has a frequency step of 156 Hz, printed below, so S2 uses a 250 ms Hann window with 90%
overlap (4 Hz steps), a standard choice. Bursts are epochs where the band's power is above halfway
between its minimum and maximum in W (claims file). "Power around 2 Hz" is a local maximum of the
time-averaged spectrum within 1 to 3 Hz.

In [ ]:
import scipy.signal as sps
f_default = sps.spectrogram(np.zeros(92000), fs=m.FS, nperseg=256, noverlap=128, nfft=256)[0]
print("NetPyNE fft defaults: frequency step", f_default[1], "Hz; bins below 100 Hz:", f_default[f_default < 100].tolist())

TF = {}
for c, d in D.items():
    x = d["lfp"][int(2.0 * m.FS):int(4.3 * m.FS)]
    f1, t1, s1 = m.morlet_spec(x, m.FS)
    f2, t2, s2 = m.stft_spec(x, m.FS)
    TF[(c, "S1")] = (f1, t1, s1, 1 / m.FS)
    TF[(c, "S2")] = (f2, t2, s2, t2[1] - t2[0])

def spec_row(c, s, bands):
    f, t, S, dt = TF[(c, s)]
    tm = S.mean(axis=1)
    loc = [float(f[i]) for i in range(1, len(f) - 1) if tm[i] > tm[i - 1] and tm[i] > tm[i + 1] and 1 <= f[i] <= 3]
    row = {"file": m.CONDITIONS[c], "reading": s, "local max 1-3 Hz": loc,
           "mean 15-20 Hz power": m.band_power(f, S, (15, 20)).mean()}
    for band in bands:
        b = m.band_bursts(t, m.band_power(f, S, band), dt)
        row[f"{band} n"] = b["n"]
        row[f"{band} rate Hz"] = round(b["rate"], 2)
        row[f"{band} median ms"] = round(b["median_ms"], 1)
        row[f"{band} min-max ms"] = (round(b["durations"].min() * 1e3), round(b["durations"].max() * 1e3)) if b["n"] else None
        row[f"{band} frac above"] = round(b["frac_above"], 2)
    return row

fig, ax = plt.subplots(1, 4, figsize=(16, 3))
for k, c in enumerate(D):
    f, t, S, dt = TF[(c, "S1")]
    ax[k].imshow(S[:, ::400], aspect="auto", origin="lower", extent=[2.0, 4.3, f[0], f[-1]], cmap="jet")
    ax[k].set_title(m.CONDITIONS[c] + " (S1)"); ax[k].set_xlabel("s"); ax[k].set_ylabel("Hz")
    print(m.CONDITIONS[c], "S1 plotted power range", S.min(), S.max())
plt.tight_layout(); plt.show()

#### fig3a-control-rest-spectrogram

Reproduced when the 2 Hz peak is present and the 15 to 25 Hz burst rate is within 1.7 +/- 0.17 Hz under every reading.

In [ ]:
t3a = pd.DataFrame([spec_row("01", s, [(15, 25)]) for s in ("S1", "S2")])
print(t3a.to_string(index=False))
for _, r in t3a.iterrows():
    print(r["reading"], "2 Hz peak present:", len(r["local max 1-3 Hz"]) > 0, "; burst rate within 1.53-1.87:", within(r["(15, 25) rate Hz"], (1.7, 0.17)))

**Result: not reproduced.** Neither spectrogram reading has a local maximum of time-averaged power within 1 to 3 Hz (S1 decreases from 1 Hz), and the 15 to 25 Hz burst rate is 1.30 Hz (S1) and 0.96 Hz (S2), outside 1.53 to 1.87 Hz.

#### fig3b-control-activated-spectrogram

Reproduced when, under every reading, the activated spectrum has no local maximum within 1 to 3 Hz, mean 15 to 20 Hz power is lower than at rest, the 14 to 16 Hz burst rate is 1.5 +/- 0.15 Hz, the 25 to 35 Hz burst rate 1.0 +/- 0.1 Hz, its median duration 50 +/- 5 ms, and the 14 to 16 Hz burst durations fall within 90 to 550 ms.

In [ ]:
t3b = pd.DataFrame([spec_row("03", s, [(14, 16), (25, 35)]) for s in ("S1", "S2")])
print(t3b.to_string(index=False))
for _, r in t3b.iterrows():
    s = r["reading"]
    rest15 = spec_row("01", s, [])["mean 15-20 Hz power"]
    f, t, S, dt = TF[("03", s)]
    d15 = m.band_bursts(t, m.band_power(f, S, (14, 16)), dt)["durations"] * 1e3
    parts = {"no 1-3 Hz max": len(r["local max 1-3 Hz"]) == 0, "15-20 Hz lower than rest": r["mean 15-20 Hz power"] < rest15,
             "15 Hz burst rate": within(r["(14, 16) rate Hz"], (1.5, 0.15)), "25-35 Hz burst rate": within(r["(25, 35) rate Hz"], (1.0, 0.1)),
             "25-35 Hz median duration": within(r["(25, 35) median ms"], (50, 5)),
             "15 Hz durations in 90-550 ms": bool(np.all((d15 >= 90) & (d15 <= 550)))}
    print(s, f"15-20 Hz power activated/rest = {r['mean 15-20 Hz power'] / rest15:.2f};", parts)

**Result: not reproduced.** Two parts hold under both readings: no 1 to 3 Hz maximum, and 15 to 20 Hz power lower than at rest (ratio 0.42 under S1, 0.57 under S2). The 14 to 16 Hz burst rate is 1.30 and 2.89 Hz (expected 1.5 +/- 0.15), the 25 to 35 Hz burst rate 1.30 and 0.96 Hz (within only under S2), its median duration 66.9 and 100 ms (expected 50 +/- 5), and the 14 to 16 Hz burst durations span 155 to 557 ms under S1 and 50 to 300 ms under S2, each outside 90 to 550 ms at one end.

#### fig6a-pd-rest-spectrogram

Reproduced when the 25 to 35 Hz burst rate is 5.0 +/- 0.5 Hz and the burst durations fall within 45 to 275 ms under every reading; the share of time the 14 to 16 Hz power is above half-height is reported.

In [ ]:
t6a = pd.DataFrame([spec_row("02", s, [(25, 35), (14, 16)]) for s in ("S1", "S2")])
print(t6a.to_string(index=False))
for _, r in t6a.iterrows():
    s = r["reading"]
    f, t, S, dt = TF[("02", s)]
    d = m.band_bursts(t, m.band_power(f, S, (25, 35)), dt)["durations"] * 1e3
    print(s, "rate within 4.5-5.5:", within(r["(25, 35) rate Hz"], (5.0, 0.5)), "; durations in 45-275 ms:", bool(np.all((d >= 45) & (d <= 275))),
          "; 14-16 Hz above half-height for", r["(14, 16) frac above"])

**Result: not reproduced.** The 25 to 35 Hz burst rate is 1.74 Hz (S1) and 1.45 Hz (S2), against 4.5 to 5.5 Hz, and burst durations reach outside 45 to 275 ms (42 ms under S1, 425 ms under S2). The 14 to 16 Hz power is above its half-height 83% of the time under S1 and 36% under S2.

#### fig6b-pd-activated-spectrogram

Reproduced when, under every reading, the 20 to 35 Hz burst rate is within 9 to 22 Hz and the median burst duration within 22.5 to 110 ms.

In [ ]:
t6b = pd.DataFrame([spec_row("04", s, [(20, 35)]) for s in ("S1", "S2")])
print(t6b.to_string(index=False))
for _, r in t6b.iterrows():
    print(r["reading"], "rate in 9-22 Hz:", 9 <= r["(20, 35) rate Hz"] <= 22, "; median in 22.5-110 ms:", 22.5 <= r["(20, 35) median ms"] <= 110)

**Result: not reproduced.** The 20 to 35 Hz burst rate is 2.61 Hz (S1) and 2.89 Hz (S2), outside 9 to 22 Hz; the median burst duration, 65.1 and 75 ms, is within 22.5 to 110 ms.

### control-coincident-spikes and pd-coincident-spikes

> control-coincident-spikes: In the control condition 1% of PT5B spikes are coincident with another
> PT5B spike within 1 ms at rest, and 3% in the activated state.
>
> pd-coincident-spikes: In the parkinsonian condition 2% of PT5B spikes are coincident (2 or more
> spikes in 1 ms) at rest and 3% in the activated state.

Readings (a) a spike of another PT5B cell within 1 ms either side or (b) in the same 1 ms bin;
windows (i) W, (ii) 0.3 to 4.3 s, (iii) 0 to 4.0 s. Reproduced when under one combination both
values are within tolerance. As a variant, not a reading of the claims file, the same share among
the 20 PT5B cells whose potentials are recorded.

In [ ]:
EXP["control-coincident-spikes"] = [(1, 0.1), (3, 0.3)]
EXP["pd-coincident-spikes"] = [(2, 0.2), (3, 0.3)]
COINC = {}
for cid, (rest, act) in {"control-coincident-spikes": ("01", "03"), "pd-coincident-spikes": ("02", "04")}.items():
    print(cid, "expected", [e[0] for e in EXP[cid]])
    for rd in "ab":
        for wn, w in [("(i) W", m.WINDOW), ("(ii) 0.3-4.3 s", (0.3, 4.3)), ("(iii) 0-4.0 s", (0.0, 4.0))]:
            res = [m.coincident_fraction(D[c], "PT5B", w, rd) for c in (rest, act)]
            vals = [r[0] for r in res]
            COINC[(cid, rd, wn)] = vals
            print(f"  ({rd}) {wn:15s} {np.round(vals, 3).tolist()} %  spikes {[r[1] for r in res]}  within {[bool(within(v, e)) for v, e in zip(vals, EXP[cid])]}")
    for c in (rest, act):
        sub = dict(D[c]); sub["pop"] = {g: ("SUB" if g in D[c]["vm_gids"] else p) for g, p in D[c]["pop"].items()}
        print(f"  variant, the 20 recorded cells, {m.CONDITIONS[c]}: (a) W {m.coincident_fraction(sub, 'SUB', m.WINDOW, 'a')[0]:.2f}%, "
              f"(b) W {m.coincident_fraction(sub, 'SUB', m.WINDOW, 'b')[0]:.2f}%")
rate_ms = {m.CONDITIONS[c]: round(len(m.pop_spikes(D[c], 'PT5B')) / 4300, 1) for c in D}
print("PT5B spikes per ms over the whole population:", rate_ms)

**Result: not reproduced.** Under every reading 99.99% to 100% of control PT5B spikes have a spike of another PT5B cell within 1 ms, against 1% and 3%: the 1,435 PT5B cells fire 17 to 42 spikes per ms together. Among only the 20 recorded cells the share is still 24.8% to 72.8%. The paper's 1% to 3% cannot come from all PT5B cells with this definition.

**Result: not reproduced.** Under every reading 99.95% to 100% of parkinsonian PT5B spikes are coincident, against 2% and 3%; among the 20 recorded cells, 27.0% to 72.2%.

### Unitary events: fig4-control-unitary-events, fig7-pd-unitary-events, rest-synchrony-increase

> fig4-control-unitary-events: In the control condition, PT5B coincident firing above that expected
> from rate-matched Poisson processes occurs at rest in 100 to 275 ms clusters with 15 to 24 Hz
> periodicity (mean 19 Hz), totalling 0.7 of 4.0 s (17.5%), and in the activated state in more
> frequent 125 to 250 ms clusters totalling 1.1 of 4.0 s (27.5%).
>
> fig7-pd-unitary-events: In the parkinsonian condition, significant PT5B coincident firing occurs
> at rest in 400 to 1100 ms clusters of 6 to 16 periods at 14 to 15 Hz (mean 15 Hz), totalling 3.1
> of 4.0 s (77.5%), and in the activated state in 100 to 350 ms epochs totalling 1.4 of 4.0 s (35.0%).
>
> rest-synchrony-increase: At rest, PT5B spike synchrony is substantially larger in the
> parkinsonian than in the control condition (significant synchrony 3.1 against 0.7 of 4.0 s).

Method of the claims file: coincidences are 1 ms bins with 2 or more PT5B spikes; expected count
from Poisson cells at their rates in the 100 ms window; a window is significant when the
joint-surprise p < 0.05; significant periods are the union of significant windows. Readings:
windows (a) slid in 1 ms steps or (b) non-overlapping; span (i) 0.3 to 4.3 s or (ii) 0 to 4.0 s;
periodicity (x) SCH spectral peak or (y) inverse mean interval between SCH peaks.

In [ ]:
EXP["fig4-control-unitary-events"] = [(0.7, 0.07), (1.1, 0.11), (19, 1.9)]
EXP["fig7-pd-unitary-events"] = [(3.1, 0.31), (1.4, 0.14), (15, 1.5)]
UE = {}
for c in D:
    for sname, step in [("(a) 1 ms steps", 0.001), ("(b) non-overlapping", 0.1)]:
        for spname, span in [("(i) 0.3-4.3 s", (0.3, 4.3)), ("(ii) 0-4.0 s", (0.0, 4.0))]:
            u = m.unitary_events(D[c], "PT5B", span, step)
            per = {r: m.periodicity(m.pop_spikes(D[c], "PT5B"), u["runs"], span, reading=r) for r in "xy"}
            UE[(c, sname, spname)] = (u, per)
rows = []
for (c, sname, spname), (u, per) in UE.items():
    rows.append({"file": m.CONDITIONS[c], "windows": sname, "span": spname, "significant s": u["total_s"],
                 "clusters": len(u["clusters_ms"]), "significant windows": f"{u['n_sig_windows']}/{u['n_windows']}",
                 "share of 1 ms bins with >=2 spikes": round(u["coinc_share_bins"], 3),
                 "median observed per window": np.median(u["obs"]), "median expected": round(float(np.median(u["exp"])), 2),
                 "periodicity x": np.mean(per["x"]) if per["x"] else np.nan, "periodicity y": np.mean(per["y"]) if per["y"] else np.nan})
uet = pd.DataFrame(rows)
print(uet.to_string(index=False))
for cid, (rest, act) in {"fig4-control-unitary-events": ("01", "03"), "fig7-pd-unitary-events": ("02", "04")}.items():
    print(cid, "expected", [e[0] for e in EXP[cid]])
    for sname in ["(a) 1 ms steps", "(b) non-overlapping"]:
        for spname in ["(i) 0.3-4.3 s", "(ii) 0-4.0 s"]:
            for pr in "xy":
                ur, pr_r = UE[(rest, sname, spname)]
                ua, _ = UE[(act, sname, spname)]
                vals = [ur["total_s"], ua["total_s"], np.mean(pr_r[pr]) if pr_r[pr] else np.nan]
                print(f"  {sname:20s} {spname:14s} ({pr}) {vals} within {[bool(within(v, e)) if np.isfinite(v) else False for v, e in zip(vals, EXP[cid])]}")

**Result: not reproduced.** Under every reading no 100 ms window is significant (0 of 3,901 slid and 0 of 40 non-overlapping windows), so the significant duration is 0 s at rest and activated, against 0.7 and 1.1 s. In 95% to 100% of 1 ms bins two or more PT5B cells fire, so observed and expected coincidence counts are both about 100 per window and the test cannot find excess; the paper's analysis must have used fewer cells or another definition.

**Result: not reproduced.** As for the control files, no window is significant under any reading (0 s against 3.1 and 1.4 s), with two or more PT5B spikes in 95% to 100% of 1 ms bins.

#### rest-synchrony-increase

Reproduced when the total duration of significant synchrony at rest is larger in the parkinsonian file under every reading.

In [ ]:
for sname in ["(a) 1 ms steps", "(b) non-overlapping"]:
    for spname in ["(i) 0.3-4.3 s", "(ii) 0-4.0 s"]:
        a, b = UE[("01", sname, spname)][0]["total_s"], UE[("02", sname, spname)][0]["total_s"]
        print(f"{sname:20s} {spname:14s} control {a} s, parkinsonian {b} s, parkinsonian larger: {b > a}")

**Result: not reproduced.** The significant synchrony at rest is 0 s in both the control and the parkinsonian file under every reading, so it is not larger in the parkinsonian file. The outcome follows from the two unitary-event claims, whose method finds nothing on these data.

### fig5c-pd-pt5b-rate-activated

> In the parkinsonian condition PT5B neurons fire more in the activated state than at rest (14.2 +/- 7.1 against 29.7 +/- 12.7 spikes/s, p < 0.001).

Criterion: files -02 and -04, R1, R2 and R3; as fig2c-control-pt5b-rate-activated.

In [ ]:
EXP["fig5c-pd-pt5b-rate-activated"] = [(14.2, 1.42), (7.1, 0.71), (29.7, 2.97), (12.7, 1.27)]
check_rates("fig5c-pd-pt5b-rate-activated", "PT5B", ['02', '04'], [('02', 'mean'), ('02', 'sd'), ('04', 'mean'), ('04', 'sd')])
cmpt = compare("PT5B", "02", "04")
print()
print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.001, per reading:", (ps < 0.001).all(axis=1).to_dict())

**Result: reproduced.** Under R1 parkinsonian PT5B rates are 14.24 +/- 7.13 at rest and 29.68 +/- 12.67 spikes/s activated, and under R2 14.70 +/- 6.76 and 30.57 +/- 11.75; R3 equals R1. All four are within tolerance under every reading, and every test gives p below 1e-200.


### fig5c-pd-it23-rate-activated

> In the parkinsonian condition IT2/3 neurons fire more in the activated state than at rest (0.78 +/- 1.1 against 4.3 +/- 7.1 spikes/s, p < 0.001).

Criterion: files -02 and -04, population IT2, R1, R2 and R3; p < 0.001.

In [ ]:
EXP["fig5c-pd-it23-rate-activated"] = [(0.78, 0.078), (1.1, 0.11), (4.3, 0.43), (7.1, 0.71)]
check_rates("fig5c-pd-it23-rate-activated", "IT2", ['02', '04'], [('02', 'mean'), ('02', 'sd'), ('04', 'mean'), ('04', 'sd')])
cmpt = compare("IT2", "02", "04")
print()
print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.001, per reading:", (ps < 0.001).all(axis=1).to_dict())

**Result: partially reproduced.** Under R3 parkinsonian IT2 rates are 0.783 +/- 1.105 at rest and 4.295 +/- 7.066 activated, all four within tolerance, and the Welch test finds the increase at p = 1.7e-04, but the Mann-Whitney test gives 0.0018 and the paired tests on the 22 cells kept in both files 0.60 (Wilcoxon) and 0.21 (paired t). The outcome rests on R3. Under R1 (0.025 +/- 0.239 and 0.164 +/- 1.598) and R2 (1.163 +/- 1.173 and 6.163 +/- 7.769) the values are outside tolerance.


### pd-it5-silenced-activated

> In the parkinsonian condition IT5A and IT5B neurons fire at 2.2 and 1.2 spikes/s at rest and stop
> firing (0 spikes/s) in the activated state.

Criterion: files -02 and -04, as control-it5-silenced-activated.

In [ ]:
it5("pd-it5-silenced-activated", "02", "04", [(2.2, 0.22), (1.2, 0.12), (0, 0.05), (0, 0.05)])

**Result: reproduced.** Under R3 the parkinsonian rest rates are 2.17 (IT5A) and 1.25 spikes/s (IT5B), within tolerance of 2.2 and 1.2, and the activated rates are 0 (one IT5A and eleven IT5B units, all firing before 0.05 s). The outcome rests on R3: under R1 the rest rates are 0.47 and 0.47, and under R2 2.57 and 2.19 with the activated means undefined.


### pd-it6-rate-unchanged

> In the parkinsonian condition the IT6 firing rate does not change with activation (6.0 spikes/s at
> rest and in the activated state).

A claim of absence. Criterion: files -02 and -04, R1, R2 and R3; reproduced when both means are within
tolerance under one rate reading and no test reading finds a difference at p < 0.05; not reproduced
when a test does.

In [ ]:
EXP["pd-it6-rate-unchanged"] = [(6.0, 0.6), (6.0, 0.6)]
check_rates("pd-it6-rate-unchanged", "IT6", ["02", "04"], [("02", "mean"), ("04", "mean")])
cmpt = compare("IT6", "02", "04")
print(); print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAny test finds a difference at p < 0.05, per reading:", (ps < 0.05).any(axis=1).to_dict())
print("Difference of means (activated - rest), per reading:", (cmpt["mean 2"] - cmpt["mean 1"]).round(3).tolist())

**Result: reproduced.** Under R3 the IT6 rates are 6.00 at rest and 5.99 spikes/s activated, within tolerance of 6.0, and no test finds a difference at p < 0.05 (smallest p 0.49, difference -0.012 spikes/s). The outcome rests on R3: under R1 the means, 1.33 and 1.73, are outside tolerance and every test finds a difference (p 0.00017 to 0.019), and under R2 no test does but the means, 6.79 and 6.64, are outside 5.4 to 6.6.


### rest-pt5b-rate-pd-vs-control

> At rest, PT5B firing rate is higher and less variable in the parkinsonian than in the control
> condition (14.2 +/- 7.1 against 11.7 +/- 9.4 spikes/s, p < 0.001), a paradoxical increase given the
> reduced PT5B excitability.

Criterion: files -01 and -02, R1, R2 and R3; reproduced when under one reading the four rest values are
within tolerance, the parkinsonian mean is higher with p < 0.001 under every test (paired by gid),
and the parkinsonian standard deviation is lower.

In [ ]:
EXP["rest-pt5b-rate-pd-vs-control"] = [(11.7, 1.17), (9.4, 0.94), (14.2, 1.42), (7.1, 0.71)]
check_rates("rest-pt5b-rate-pd-vs-control", "PT5B", ["01", "02"], [("01", "mean"), ("01", "sd"), ("02", "mean"), ("02", "sd")])
cmpt = compare("PT5B", "01", "02")
print(); print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.001, per reading:", (ps < 0.001).all(axis=1).to_dict())
t = rate_rows("PT5B", ["01", "02"]).set_index(["reading", "file"])
print("Parkinsonian sd lower, per reading:", {r: bool(t.loc[(r, "02"), "sd"] < t.loc[(r, "01"), "sd"]) for r in ["R1", "R2", "R3"]})

**Result: reproduced.** Under R1 the rest PT5B rate is 11.69 +/- 9.40 in the control and 14.24 +/- 7.13 spikes/s in the parkinsonian file, all four within tolerance; every test gives p below 1e-15, and the standard deviation is lower in the parkinsonian file. R2 and R3 also meet the criterion (R3 equals R1).


### rest-it4-rate-pd-vs-control

> At rest, IT4 firing rate is higher in the parkinsonian than in the control condition (14.1 against 2.6 spikes/s, p < 0.05).

Criterion: files -01 and -02, R1, R2 and R3; reproduced when under one reading both means are within tolerance and the parkinsonian rate is higher with p < 0.05 under every test.

In [ ]:
EXP["rest-it4-rate-pd-vs-control"] = [(2.6, 0.26), (14.1, 1.41)]
check_rates("rest-it4-rate-pd-vs-control", "IT4", ['01', '02'], [('01', 'mean'), ('02', 'mean')])
cmpt = compare("IT4", "01", "02")
print()
print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.05, per reading:", (ps < 0.05).all(axis=1).to_dict())

**Result: not reproduced.** Under R3 the IT4 means are 2.57 (11 cells) in the control and 14.13 spikes/s (2 cells) in the parkinsonian file, both within tolerance, but no test finds the increase at p < 0.05 (Mann-Whitney 0.077, Welch 0.31, paired tests 1.0 and 0.68 on 2 cells). Under R1 both means are 0.037 spikes/s, and under R2 3.14 and 14.13, the control one outside tolerance, with no test below 0.05 either. No reading meets the criterion.


### rest-it5a-rate-pd-vs-control

> At rest, IT5A firing rate is lower in the parkinsonian than in the control condition (2.2 against 3.3 spikes/s, p < 0.01).

Criterion: files -01 and -02, R1, R2 and R3; reproduced when under one reading both means are within tolerance and the parkinsonian rate is lower with p < 0.01 under every test.

In [ ]:
EXP["rest-it5a-rate-pd-vs-control"] = [(3.3, 0.33), (2.2, 0.22)]
check_rates("rest-it5a-rate-pd-vs-control", "IT5A", ['01', '02'], [('01', 'mean'), ('02', 'mean')])
cmpt = compare("IT5A", "01", "02")
print()
print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.01, per reading:", (ps < 0.01).all(axis=1).to_dict())

**Result: reproduced.** Under R3 the rest IT5A rates are 3.25 in the control and 2.16 spikes/s in the parkinsonian file, within tolerance of 3.3 and 2.2, and every test finds the decrease at p below 0.01 (largest 0.0027, Welch). The outcome rests on R3: R1 gives 1.21 and 0.47, outside tolerance, and R2 3.57 and 2.57, the parkinsonian value outside 1.98 to 2.42, with unpaired tests 0.017 and 0.012.


### activated-pt5b-rate-pd-vs-control

> In the activated state, PT5B firing rate is higher and less variable in the parkinsonian than in
> the control condition (29.7 +/- 12.7 against 26.8 +/- 18.2 spikes/s, p < 0.001).

Criterion: files -03 and -04, as rest-pt5b-rate-pd-vs-control. The Discussion says there was no
significant change in the activated state; the outcome under every test is reported for both.

In [ ]:
EXP["activated-pt5b-rate-pd-vs-control"] = [(26.8, 2.68), (18.2, 1.82), (29.7, 2.97), (12.7, 1.27)]
check_rates("activated-pt5b-rate-pd-vs-control", "PT5B", ["03", "04"], [("03", "mean"), ("03", "sd"), ("04", "mean"), ("04", "sd")])
cmpt = compare("PT5B", "03", "04")
print(); print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.001, per reading:", (ps < 0.001).all(axis=1).to_dict())
print("Any test at p < 0.05 (the Discussion's 'no significant change'), per reading:", (ps < 0.05).any(axis=1).to_dict())
t = rate_rows("PT5B", ["03", "04"]).set_index(["reading", "file"])
print("Parkinsonian sd lower, per reading:", {r: bool(t.loc[(r, "04"), "sd"] < t.loc[(r, "03"), "sd"]) for r in ["R1", "R2", "R3"]})

**Result: partially reproduced.** Under R1 (and R3, which equals it) the activated PT5B rate is 26.76 +/- 18.17 in the control and 29.68 +/- 12.67 spikes/s in the parkinsonian file, all four within tolerance, and the standard deviation is lower; the Wilcoxon, paired t and Welch tests give p from 1.6e-08 to 6.5e-07, but the Mann-Whitney test gives 0.0052, above 0.001. Under R2 the means are 30.60 and 30.57 and no test is below 0.05. The Discussion's statement of no significant change in the activated state holds under R2 and not under R1 or R3.


### activated-it6-rate-pd-vs-control

> In the activated state, IT6 firing rate is higher in the parkinsonian than in the control condition (6.0 against 3.5 spikes/s, p < 0.001).

Criterion: files -03 and -04, R1, R2 and R3; reproduced when under one reading both means are within tolerance and the parkinsonian rate is higher with p < 0.001 under every test.

In [ ]:
EXP["activated-it6-rate-pd-vs-control"] = [(3.5, 0.35), (6.0, 0.6)]
check_rates("activated-it6-rate-pd-vs-control", "IT6", ['03', '04'], [('03', 'mean'), ('04', 'mean')])
cmpt = compare("IT6", "03", "04")
print()
print(cmpt.to_string(index=False))
ps = cmpt.set_index("reading")[["T1 Wilcoxon", "T2 paired t", "T3 Mann-Whitney", "T4 Welch t"]]
print("\nAll four tests below p = 0.001, per reading:", (ps < 0.001).all(axis=1).to_dict())

**Result: reproduced.** Under R3 the activated IT6 rates are 3.47 in the control and 5.99 spikes/s in the parkinsonian file, within tolerance of 3.5 and 6.0, and every test finds the increase at p below 1e-07 (largest 3.4e-08). The outcome rests on R3: R1 gives 0.95 and 1.73 and R2 4.87 and 6.64, outside tolerance, though the parkinsonian rate is higher under both.


### rest-beta-power-order-of-magnitude

> At rest, LFP beta power around 15 Hz is an order of magnitude larger in the parkinsonian than in
> the control condition.

Criterion: files -01 and -02, LFP in W, readings S1, S2 (E1 only); power (a) at 15 Hz and (b) mean
over 14 to 16 Hz; ratio of time-averaged parkinsonian to control power; reproduced when the ratio is
in 3.16 to 31.6 under every reading, not reproduced when below 3.16 under every reading. As a
variant, the ratio of the spectrograms' maxima (the colour bars of Figs. 3 and 6 are x 10^-5 and x
10^-4).

In [ ]:
rows = []
for s in ("S1", "S2"):
    f1, _, S1c, _ = TF[("01", s)]
    f2, _, S2p, _ = TF[("02", s)]
    at15 = lambda f, S: S[np.argmin(np.abs(f - 15))].mean()
    for name, fn in [("(a) at 15 Hz", at15), ("(b) 14-16 Hz", lambda f, S: m.band_power(f, S, (14, 16)).mean())]:
        a, b = fn(f1, S1c), fn(f2, S2p)
        rows.append({"reading": s, "measure": name, "control": a, "parkinsonian": b, "ratio": b / a,
                     "log10 ratio": np.log10(b / a), "in 3.16-31.6": 10 ** 0.5 <= b / a <= 10 ** 1.5})
    rows.append({"reading": s, "measure": "variant, max over the spectrogram", "control": S1c.max(), "parkinsonian": S2p.max(),
                 "ratio": S2p.max() / S1c.max(), "log10 ratio": np.log10(S2p.max() / S1c.max()),
                 "in 3.16-31.6": 10 ** 0.5 <= S2p.max() / S1c.max() <= 10 ** 1.5})
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** Parkinsonian rest LFP power around 15 Hz is far larger than control under every reading, but by more than the criterion's order of magnitude: the ratio is 129 (S1, at 15 Hz), 111 (S1, 14 to 16 Hz) and 65 (S2, both), above 31.6, so neither the criterion's reproduced range (3.16 to 31.6) nor its not-reproduced range (below 3.16) applies. The ratio of the spectrograms' maxima, a variant, is 21.2 under S1 and 3.0 under S2.

### activated-20-35hz-power-pd-vs-control

> In the activated state, the parkinsonian condition shows more power at 20 to 35 Hz than the
> control condition.

Criterion: files -03 and -04, LFP readings S1, S2 (E1) and the PT5B SCH spectrum; reproduced when
mean 20 to 35 Hz power is higher in the parkinsonian file under every reading.

In [ ]:
rows = []
for s in ("S1", "S2"):
    a = m.band_power(TF[("03", s)][0], TF[("03", s)][2], (20, 35)).mean()
    b = m.band_power(TF[("04", s)][0], TF[("04", s)][2], (20, 35)).mean()
    rows.append({"reading": f"LFP {s}", "control": a, "parkinsonian": b, "ratio": b / a, "higher": b > a})
f, p3 = SPEC["03"]; _, p4 = SPEC["04"]
sel = (f >= 20) & (f <= 35)
rows.append({"reading": "PT5B SCH (Welch)", "control": p3[sel].mean(), "parkinsonian": p4[sel].mean(), "ratio": p4[sel].mean() / p3[sel].mean(), "higher": p4[sel].mean() > p3[sel].mean()})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Mean 20 to 35 Hz power is higher in the parkinsonian activated file under every reading: by a factor 4.55 for the LFP under S1, 4.71 under S2, and 11.8 for the PT5B SCH spectrum.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1-network-size | not reproduced (only cells that fired are stored) |
| model-pt5b-excitability-decrease | blocked: data_missing |
| robustness-four-sets | blocked: methods_underspecified |
| fig2-control-rest-beta | reproduced |
| fig2-control-activated-gamma | reproduced |
| fig2c-control-pt5b-rate-activated | reproduced |
| fig2c-control-it23-rate-activated | partially reproduced (under R3; paired tests 0.011 and 0.016) |
| fig2c-control-it6-rate-activated | reproduced (under R3) |
| control-it5-silenced-activated | reproduced (under R3) |
| control-it4-rate-activated | reproduced (under R3) |
| pt5b-highest-rate | blocked: methods_underspecified (population readings disagree) |
| control-cv-isi | reproduced |
| control-subthreshold-oscillations | blocked: methods_underspecified |
| control-duty-cycle | not reproduced |
| fig3a-control-rest-spectrogram | not reproduced |
| fig3b-control-activated-spectrogram | not reproduced |
| control-coincident-spikes | not reproduced |
| fig4-control-unitary-events | not reproduced |
| fig5-pd-rest-beta | reproduced |
| fig5-pd-activated-gamma | not reproduced |
| fig5c-pd-pt5b-rate-activated | reproduced |
| fig5c-pd-it23-rate-activated | partially reproduced (under R3; only the Welch test below 0.001) |
| pd-it5-silenced-activated | reproduced (under R3) |
| pd-it6-rate-unchanged | reproduced (under R3) |
| pd-cv-isi | reproduced |
| pd-no-subthreshold-oscillations | blocked: methods_underspecified |
| pd-activated-vm-more-negative | reproduced |
| pd-duty-cycle | not reproduced |
| fig6a-pd-rest-spectrogram | not reproduced |
| fig6b-pd-activated-spectrogram | not reproduced |
| pd-coincident-spikes | not reproduced |
| fig7-pd-unitary-events | not reproduced |
| rest-pt5b-rate-pd-vs-control | reproduced |
| rest-it4-rate-pd-vs-control | not reproduced (values match under R3, no test below 0.05) |
| rest-it5a-rate-pd-vs-control | reproduced (under R3) |
| activated-pt5b-rate-pd-vs-control | partially reproduced |
| activated-it6-rate-pd-vs-control | reproduced (under R3) |
| rest-beta-power-order-of-magnitude | partially reproduced |
| activated-20-35hz-power-pd-vs-control | reproduced |
| rest-synchrony-increase | not reproduced |

**How the analysis differs from the paper's.** The files hold one LFP trace, so only electrode
reading E1 was run, taking "LFP 0" as electrode 0; E2 (mean of 11 electrodes) could not be. The
spectrogram reading S2 uses a 250 ms window, since NetPyNE's own fft defaults cannot resolve
below 156 Hz. Population sizes for R1 come from the gid blocks, since silent cells are not stored.
The paper's IT rates match exactly when the mean is taken over the cells that fired at any time
in the simulation (R3, the cells the file holds), and not over all cells (R1) or the cells active
in W (R2). R3 was added to the claims file's readings in answer to a question; the paper does not
say which cells enter a mean, so the nine IT outcomes that hold only under R3 rest on it as an
assumption. The coincidence and
unitary-event analyses follow the claims file's definitions over all PT5B cells; with 17 to 42
PT5B spikes per ms these saturate, so the paper must have used another set of cells or another
definition, which it does not state.

**What was found about the dataset.** The units table holds only the cells that fired (2,917 to
4,480 of 10,073 M1 cells in the paper's set), where the description says spike trains of all
10,073 neurons. The paper's set holds 20 PT5B potentials per file, not 400, and those potentials
match their cells' stored spikes only in the first half second; in W they come from a run that
diverged from the stored spikes. `sub-001/sub-001.nwb` duplicates
`sub-001_ses-sM1-04-22-2024-01.nwb`. Each file holds one LFP trace with no electrode position.
The files' related publication is the preprint, the dandiset lists no related resource, and its
summary names spike sorting as the technique of a simulation.